# ML-07 — Baseline Action Score and Top-20 Review

This is a transparent, decision-support baseline for the Refresh / Content Opportunity Scoring lane.

## 1. My rule and its reason codes

The rule is: rank pages by staleness plus volume. Staleness is days since the recorded content update; volume is the mean GSC impressions in the trailing seven days. I add their percentile ranks, with no fitted weights. Higher is first in the editor queue.

For the warehouse, the decision date is 2026-03-31. The volume feature is the Week 3 feature feat_impr_7d, a trailing seven-day mean of gsc_impressions. The April window is held out for evaluation only. The one action label is editor_review_for_refresh and the one reason code is staleness_plus_volume. Negative or mixed checks below are reported plainly, not tuned away.

In [1]:
from pathlib import Path
import json, os, tempfile
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score


def repo_root():
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / 'data' / 'raw' / 'content_refresh_anonymized.csv').exists():
            return p
    raise FileNotFoundError('Run from inside ML-INTERNSHIP.')


ROOT = repo_root()
OUT = ROOT / 'work' / 'outputs'
OUT.mkdir(parents=True, exist_ok=True)
DECISION_DATE = pd.Timestamp('2026-03-31')


def token_from_existing_w03_setup():
    try:
        from google.colab import userdata
        return userdata.get('HF_TOKEN').strip()
    except (ImportError, KeyError, AttributeError):
        try:
            from dotenv import load_dotenv
            load_dotenv(ROOT / '.env')
        except ImportError:
            pass
        return os.environ.get('HF_TOKEN', '').strip()


def warehouse_file(filename, token):
    from huggingface_hub import hf_hub_download
    try:
        return hf_hub_download('FlyRank/internship-warehouse', filename=filename, repo_type='dataset', token=token)
    except PermissionError:
        return hf_hub_download('FlyRank/internship-warehouse', filename=filename, repo_type='dataset', token=token, local_dir=Path(tempfile.gettempdir()) / 'flyrank_w04_warehouse')


def warehouse_frame():
    token = token_from_existing_w03_setup()
    if not token:
        raise RuntimeError('HF_TOKEN is not configured')
    dim = pd.read_parquet(warehouse_file('dim_content.parquet', token), columns=['client_hash_id', 'content_hash_id', 'content_updated_date'])
    cols = ['report_date', 'client_hash_id', 'content_hash_id', 'gsc_data_available', 'gsc_impressions']
    mar = pd.read_parquet(warehouse_file('fact_content_daily_performance/month=2026-03/data_0.parquet', token), columns=cols)
    apr = pd.read_parquet(warehouse_file('fact_content_daily_performance/month=2026-04/data_0.parquet', token), columns=cols)
    for x in [mar, apr]:
        x['report_date'] = pd.to_datetime(x['report_date'])
    mar = mar[mar['gsc_data_available'].astype(bool)].copy()
    apr = apr[apr['gsc_data_available'].astype(bool)].copy()
    keys = ['client_hash_id', 'content_hash_id']
    prior = mar.groupby(keys, as_index=False).agg(march_impressions_30d=('gsc_impressions', 'sum'))
    volume = mar[mar['report_date'].ge('2026-03-25')].groupby(keys, as_index=False).agg(feat_impr_7d=('gsc_impressions', 'mean'))
    future = apr.groupby(keys, as_index=False).agg(april_impressions_30d=('gsc_impressions', 'sum'))
    dim['content_updated_date'] = pd.to_datetime(dim['content_updated_date'], errors='coerce')
    x = prior.merge(volume, on=keys).merge(future, on=keys).merge(dim, on=keys, how='left')
    x['days_since_last_update'] = (DECISION_DATE - x['content_updated_date']).dt.days
    x = x[x['days_since_last_update'].ge(0)].copy()
    x['observed_forward_decline'] = (x['march_impressions_30d'].gt(0) & x['april_impressions_30d'].lt(.8 * x['march_impressions_30d'])).astype(int)
    x['volume_signal'] = x['feat_impr_7d']
    x['data_source'] = 'warehouse: March features, April held-out label'
    return x, 'April impressions are more than 20% below March'


try:
    analysis, label_definition = warehouse_frame()
    print('Using warehouse data for the actual score and evaluation.')
except Exception as err:
    analysis = pd.read_csv(ROOT / 'data' / 'raw' / 'content_refresh_anonymized.csv')
    analysis['observed_forward_decline'] = analysis['trend_direction'].eq('down').astype(int)
    analysis['volume_signal'] = analysis['impressions_90d']
    analysis['data_source'] = 'starter CSV fallback'
    label_definition = 'starter proxy: trend_direction equals down'
    print('Warehouse unavailable; starter fallback used: {}'.format(type(err).__name__))

print('Eligible rows: {:,}'.format(len(analysis)))
print('Observed decline base rate: {:.3f}'.format(analysis['observed_forward_decline'].mean()))


def bucket_table(column):
    b = pd.qcut(analysis[column], q=5, duplicates='drop')
    return analysis.assign(bucket=b).groupby('bucket', observed=True).agg(n=('observed_forward_decline', 'size'), decline_rate=('observed_forward_decline', 'mean')).reset_index()


def verdict(table):
    r = table['decline_rate'].to_numpy()
    if len(r) < 2 or r.max() - r.min() < .02: return 'FALSE'
    d = np.diff(r)
    if np.all(d >= 0) and r[-1] - r[0] >= .02: return 'CONFIRMED'
    if np.all(d <= 0) and r[-1] - r[0] <= -.02: return 'OPPOSITE'
    return 'MIXED'

staleness_check = bucket_table('days_since_last_update')
volume_check = bucket_table('volume_signal')
print('\nStaleness vs. decline rate')
print(staleness_check.to_string(index=False, formatters={'decline_rate': '{:.3f}'.format}))
print('Verdict: {}'.format(verdict(staleness_check)))
print('Note: qcut requested five buckets but returned {} because the staleness distribution is heavily right-skewed.'.format(len(staleness_check)))
print('Most rows ({:,} of {:,}) collapse into the first bucket, while only the long tail ({:,} rows) is in the other bucket; this is a methodological detail, not a bug.'.format(int(staleness_check.loc[0, 'n']), int(staleness_check['n'].sum()), int(staleness_check.loc[1, 'n'])))
print('\nVolume vs. decline rate')
print(volume_check.to_string(index=False, formatters={'decline_rate': '{:.3f}'.format}))
print('Verdict: {}'.format(verdict(volume_check)))

Using warehouse data for the actual score and evaluation.
Eligible rows: 23,936
Observed decline base rate: 0.571

Staleness vs. decline rate
       bucket     n decline_rate
(6.999, 34.0] 22690        0.576
(34.0, 303.0]  1246        0.478
Verdict: OPPOSITE
Note: qcut requested five buckets but returned 2 because the staleness distribution is heavily right-skewed.
Most rows (22,690 of 23,936) collapse into the first bucket, while only the long tail (1,246 rows) is in the other bucket; this is a methodological detail, not a bug.

Volume vs. decline rate
          bucket    n decline_rate
  (0.999, 2.667] 4875        0.520
  (2.667, 7.143] 4756        0.574
 (7.143, 18.143] 4751        0.602
  (18.143, 47.0] 4770        0.607
(47.0, 5828.429] 4784        0.552
Verdict: MIXED


## 2. Build the ranked queue (writes the CSV)

The score is frozen before looking at the held-out label: staleness_rank plus volume_rank. The full CSV excludes April totals and the observed label, so it remains an action queue rather than an answer key.

In [2]:
analysis['staleness_rank'] = analysis['days_since_last_update'].rank(pct=True, method='average')
analysis['volume_rank'] = analysis['volume_signal'].rank(pct=True, method='average')
analysis['baseline_score'] = analysis['staleness_rank'] + analysis['volume_rank']
analysis['action_label'] = 'editor_review_for_refresh'
analysis['reason_code'] = 'staleness_plus_volume'
ranked = analysis.sort_values(['baseline_score', 'volume_signal'], ascending=False).reset_index(drop=True)
ranked.insert(0, 'queue_rank', np.arange(1, len(ranked) + 1))
id_columns = [c for c in ['client_hash_id', 'content_hash_id', 'content_id'] if c in ranked.columns]
queue_columns = ['queue_rank', 'data_source', *id_columns, 'days_since_last_update', 'volume_signal', 'staleness_rank', 'volume_rank', 'baseline_score', 'action_label', 'reason_code']
queue_path = OUT / 'baseline_action_score.csv'
ranked[queue_columns].to_csv(queue_path, index=False)
ks = [10, 20, 50, 100, 500]
precision = {str(k): float(ranked.head(k)['observed_forward_decline'].mean()) for k in ks if k <= len(ranked)}
try:
    auc = float(roc_auc_score(ranked['observed_forward_decline'], ranked['baseline_score']))
except ValueError:
    auc = None
metrics = {'data_source': ranked['data_source'].iloc[0], 'eligible_rows': int(len(ranked)), 'label_definition': label_definition, 'base_rate': float(ranked['observed_forward_decline'].mean()), 'rule': 'staleness_rank + volume_rank', 'reason_code': 'staleness_plus_volume', 'precision_at_k': precision, 'roc_auc': auc}
metrics_path = OUT / 'baseline_score_metrics.json'
metrics_path.write_text(json.dumps(metrics, indent=2), encoding='utf-8')
print('Wrote {:,} ranked rows to {}'.format(len(ranked), queue_path))
print('Base rate: {:.3f}'.format(metrics['base_rate']))
for k, value in precision.items(): print('Precision@{}: {:.3f}'.format(k, value))
print('ROC-AUC: {}'.format('not applicable' if auc is None else '{:.3f}'.format(auc)))

Wrote 23,936 ranked rows to C:\INTERNSHIP\ML-INTERNSHIP\work\outputs\baseline_action_score.csv
Base rate: 0.571
Precision@10: 0.700
Precision@20: 0.600
Precision@50: 0.480
Precision@100: 0.390
Precision@500: 0.338
ROC-AUC: 0.506


## 3. Top-20 review

The required top 10 are reviewed below. The action and rationale use only the frozen score. The later observed label is shown as an audit result, not as a reason for the rank.

In [3]:
top10 = ranked.head(10).copy()
for _, row in top10.iterrows():
    identifier = str(row[id_columns[-1]])
    days = row['days_since_last_update']
    volume = row['volume_signal']
    why = 'staleness {:.0f} days and 7-day mean impressions {:,.1f}; both ranks are high'.format(days, volume)
    if row['observed_forward_decline'] == 0:
        wrong = ('Already falsified by the held-out result: despite {:.0f} days since update and {:,.1f} mean 7-day impressions, April did not decline by more than 20%.'.format(days, volume))
    else:
        stale_margin = days - 34
        volume_margin = volume - 47
        wrong = ('Would be wrong if an audit moves the update date from {:.0f} days into the 34-days-or-less bucket, or shows the {:,.1f} mean 7-day volume is not real; this row is {:.0f} days and {:,.1f} impressions above those cutoffs.'.format(days, volume, stale_margin, volume_margin))
    print('#{} — {}\nAction: editor review for refresh. Why: {}. Falsification: {}\nObserved later decline: {}\n'.format(int(row['queue_rank']), identifier, why, wrong, int(row['observed_forward_decline'])))

#1 — content_ac4e2d9d3bbb06de
Action: editor review for refresh. Why: staleness 124 days and 7-day mean impressions 1,649.3; both ranks are high. Falsification: Already falsified by the held-out result: despite 124 days since update and 1,649.3 mean 7-day impressions, April did not decline by more than 20%.
Observed later decline: 0

#2 — content_66d1fffc91f4f029
Action: editor review for refresh. Why: staleness 124 days and 7-day mean impressions 1,357.7; both ranks are high. Falsification: Already falsified by the held-out result: despite 124 days since update and 1,357.7 mean 7-day impressions, April did not decline by more than 20%.
Observed later decline: 0

#3 — content_f2df5a8a9057783e
Action: editor review for refresh. Why: staleness 124 days and 7-day mean impressions 1,346.6; both ranks are high. Falsification: Would be wrong if an audit moves the update date from 124 days into the 34-days-or-less bucket, or shows the 1,346.6 mean 7-day volume is not real; this row is 90 days

## 4. Weak picks + leakage check

A weak pick is a top-ranked row whose held-out April total did not decline by more than 20%. That is a limitation of this simple operational rule, not something to hide. No product flags or future-window fields enter the score: gsc_data_available only filters measurable rows; GA4 flags are unused; April totals and the observed label are evaluation-only and excluded from the CSV queue.

The staleness check returned **OPPOSITE**: the more-stale bucket had a lower observed decline rate than the fresher bucket (0.478 vs. 0.576). This is a known tension in this baseline. Staleness remains because it is the flag-linked signal this assignment asks to test, but this data does not support pushing staler pages higher; volume is doing most of the real ranking work.

The ROC-AUC is 0.506, which is close to random and only barely better than chance across the full ranked list, even though precision@10 is stronger at 0.700. That weakness is consistent with staleness working against the rule outside the very top of the queue.


In [4]:
weak = top10[top10['observed_forward_decline'].eq(0)]
print('Weak picks in the top 10: {} of 10'.format(len(weak)))
for _, row in weak.iterrows():
    print('Rank {} is weak: high staleness and volume did not translate into a held-out decline greater than 20%.'.format(int(row['queue_rank'])))
print('\nLeakage check')
print('Score inputs only: days_since_last_update and volume_signal.')
print('gsc_data_available is a filter only; no product flag is a score input.')
print('April totals and observed_forward_decline are evaluation only and are absent from the queue CSV.')
print('PASS: no product flags, future-window data, trend_direction, or trend_pct entered the score.')

Weak picks in the top 10: 3 of 10
Rank 1 is weak: high staleness and volume did not translate into a held-out decline greater than 20%.
Rank 2 is weak: high staleness and volume did not translate into a held-out decline greater than 20%.
Rank 6 is weak: high staleness and volume did not translate into a held-out decline greater than 20%.

Leakage check
Score inputs only: days_since_last_update and volume_signal.
gsc_data_available is a filter only; no product flag is a score input.
April totals and observed_forward_decline are evaluation only and are absent from the queue CSV.
PASS: no product flags, future-window data, trend_direction, or trend_pct entered the score.


## Self-check

- [x] Every section above is filled — markdown thinking AND the code that backs it.
- [x] The notebook was run top to bottom with no errors.
- [x] No client names, URLs, private queries, or credentials appear in the notebook or outputs.
- [x] My claims use careful words: observed, measured, directional, decision-support.
- [x] Reviewed the local changes, committed, and pushed them through git not made with colab solved locally checked and than commited via git.